# Mini-Series: Memory Without Words
>> **How Neural Networks Represent and Remember Knowledge**

# PART B. External Latent Memory

# Section 1. External Latent Memory

## A first attempt: a single memory matrix

- N slots (rows), each a latent vector.

This is the simplest possible design. We will see in a moment why it is not quite right.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Memory Size
N_SLOTS: int = 8       # number of memory slots
VEC_DIM: int = 16      # dimensionality of each slot

# single memory matrix: each row is one slot
memory: torch.Tensor = torch.zeros(N_SLOTS, VEC_DIM)

# Visualize the initial memory state
fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(memory.numpy(), cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
ax.set_xlabel("Vector dimension")
ax.set_ylabel("Memory slot")
ax.set_title("Naive memory M — a single matrix  (N_SLOTS × VEC_DIM), initialized to zero")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("Shape:", memory.shape, "  — one vector per slot, but no way to separate 'what to look for' from 'what to return'")

## The problem with a single matrix

When you write one vector per slot, you mix two things that serve different purposes:

- **What you search by** — a compact descriptor that should respond to queries
- **What you retrieve** — the full content associated with that descriptor

In a library: the catalogue card is not the book. You search **cards**; you retrieve **books**. A single matrix forces both to live in the same vector, which means any change to the content corrupts the search signal, and vice versa.

The solution is exactly what transformer attention already uses: **separate key and value matrices**.

| Sub-matrix | Role | Searched by | Returned by |
|---|---|---|---|
| **K** (keys) | "catalogue cards" | query similarity | — |
| **V** (values) | "books" | — | weighted sum of latent values |

Both matrices have the same number of rows (one row = one memory slot). Keys and values for the same slot share only their row index — they live in different spaces and can have different dimensionalities.

In [ ]:
torch.manual_seed(0)

# Memory dimensions
N_SLOTS: int = 8    # number of memory slots
D_K: int = 8        # key dimensionality  (smaller — keys are compact search descriptors)
D_V: int = 16       # value dimensionality (larger  — values carry richer content)

# Two separate matrices; same number of rows, independent column spaces
mem_K: torch.Tensor = torch.zeros(N_SLOTS, D_K)   # key matrix   — what to search by
mem_V: torch.Tensor = torch.zeros(N_SLOTS, D_V)   # value matrix — what to return

# Write three synthetic facts into slots 2, 4, 6
for slot, (kv, vv) in enumerate([
    (torch.tensor([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]),  # key: "fact A"
     torch.randn(D_V)),
    (torch.tensor([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]),  # key: "fact B"
     torch.randn(D_V)),
    (torch.tensor([0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]),  # key: "fact C"
     torch.randn(D_V)),
], start=2):
    mem_K[slot] = kv
    mem_V[slot] = vv
    slot += 2  # skip a slot between facts

# visualize both matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 3))

im0 = axes[0].imshow(mem_K.numpy(), cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
axes[0].set_title(f"Key matrix K  ({N_SLOTS} × {D_K})\n— compact search descriptors")
axes[0].set_xlabel("Key dimension")
axes[0].set_ylabel("Memory slot")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(mem_V.numpy(), cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
axes[1].set_title(f"Value matrix V  ({N_SLOTS} × {D_V})\n— richer content vectors")
axes[1].set_xlabel("Value dimension")
axes[1].set_ylabel("Memory slot")
plt.colorbar(im1, ax=axes[1])

plt.suptitle("External memory as two parallel matrices — same slot index, independent spaces", y=1.02)
plt.tight_layout()
plt.show()

print(f"Key matrix shape:   {mem_K.shape}  — used only for similarity search")
print(f"Value matrix shape: {mem_V.shape}  — used only for content retrieval")

<details>
<summary>Details: <strong>Why keys and values can live in different spaces<strong></summary>

Keys exist to be compared to queries — their geometry only needs to support similarity search. A smaller key space (lower dimensionality) is fine and cheap. Values exist to encode content — they might need to carry richer, higher-dimensional information.

The model projects the controller's hidden state into key space via one learned linear map, and into value space via another. These two projections are independent: key space is optimized by gradients flowing back through the similarity computation; value space is optimized by gradients from the downstream loss on retrieved content.

The original NTM paper (Graves et al., 2014) used a single matrix — partly because the design predates standard multi-head attention notation, and partly to keep the write head simple. In practice, a split K/V memory is equivalent (you can recover the single-matrix case by using the same projection for both). We use the split form here because it makes the read/write operations structurally identical to transformer attention, and because it's what modern external-memory systems (Memorizing Transformers, ELM) actually use.
</details>

# Section 2 — Reading from Memory - Content-Based Addressing

**Reading from memory** means: given a query vector, produce a retrieved vector.

The query comes from the controller (a network processing the current input). The retrieved vector is a weighted blend of all value slots, where the weights are determined by how similar the **query** is to each **key**.

This is **content-based addressing** — you retrieve by *what you're looking for*, not by *where it lives*.

```
query q  →  similarity to each key in K  →  attention weights w  →  weighted sum of V  →  retrieved vector r
```

Three components of a read operation:
1. **Query** `q`: a vector produced by the controller, shape `(D_K,)`
2. **Attention weights** `w`: `softmax(q @ K.T / sqrt(D_K))`, shape `(N_SLOTS,)`
3. **Read result** `r`: `w @ V`, shape `(D_V,)` — a soft blend of all value slots

The attention weights over memory slots are computed exactly like transformer self-attention — scaled dot-product between the query and the key matrix, followed by softmax.

| Component | In memory terms | Shape |
|-----------|-----------------|-------|
| `q` (query) | "what I'm looking for right now" | `(D_K,)` |
| `K` (keys) | "what each slot advertises" | `(N_SLOTS, D_K)` |
| `V` (values) | "what each slot actually contains" | `(N_SLOTS, D_V)` |
| `w` (weights) | "how much to trust each slot" | `(N_SLOTS,)` |
| `r` (read result) | "what I retrieved" | `(D_V,)` |

The read result is **not** a single slot — it is a soft blend. When the weights are nearly one-hot (one slot dominates), the read is almost a hard lookup. When the weights are spread, the result interpolates between several stored facts.

In [ ]:
# Reading from split K/V memory: content-based addressing

torch.manual_seed(42)

# Memory dimensions
N_SLOTS: int = 8
D_K: int = 8    # key space — queries live here too
D_V: int = 16   # value space — richer content

# initialize memory: K stores search descriptors, V stores content
mem_K: torch.Tensor = torch.randn(N_SLOTS, D_K) * 0.1  # mostly noise (empty-ish)
mem_V: torch.Tensor = torch.randn(N_SLOTS, D_V) * 0.1

# plant a strong, interpretable fact in slot 3:
# key = "category A" direction; value = recognizable pulse: +1 in first half, -1 in second half
slot_to_write: int = 3
mem_K[slot_to_write] = F.normalize(torch.tensor([1.0, 0.8, 0.0, 0.0, 0.2, 0.0, 0.0, 0.0]), dim=0)
mem_V[slot_to_write] = torch.cat([torch.ones(D_V // 2), -torch.ones(D_V // 2)])  # structured pattern

# query: controller asks about "category A" — similar to key at slot 3
query: torch.Tensor = F.normalize(torch.tensor([0.9, 0.7, 0.1, 0.0, 0.3, 0.0, 0.0, 0.0]), dim=0)

# --- content-based read ---
# step 1: similarity scores between query and every key
scores: torch.Tensor = query @ mem_K.T / (D_K ** 0.5)        # (N_SLOTS,)
# - scale by \sqrt{D_K} — normalization to keep logits numerically well‑behaved as in transformers

# step 2: normalize to attention weights
weights: torch.Tensor = F.softmax(scores, dim=-1)            # (N_SLOTS,)
# - Converts raw similarity scores into a probability distribution over slots
#   Differentiable (unlike argmax)

# step 3: weighted blend of values
read_result: torch.Tensor = weights @ mem_V                  # (D_V,)
# - Convex combination of all value rows
#   makes the whole memory‑reading mechanism smooth, stable, and trainable.
#   - The result is a weighted average of the vectors.
#   - It lies inside the convex hull of the memory values.
#   - It cannot "explode" or produce arbitrary values.
#   - Geometry is preserved

# --- visualize ---
fig, axes = plt.subplots(1, 3, figsize=(15, 3))

# attention weights over slots (left)
axes[0].bar(range(N_SLOTS), weights.detach().numpy(), color="steelblue")
axes[0].axvline(slot_to_write, color="red", linestyle="--", linewidth=1.5, label=f"written slot {slot_to_write}")
axes[0].set_xlabel("Memory slot")
axes[0].set_ylabel("Attention weight")
axes[0].set_title("Attention weights w\nover memory slots")
axes[0].legend()

# key matrix with query similarity highlighted (middle)
# RdBu_r: Red = high similarity (strong match), Blue = low/negative similarity (opposite to query)
sim_display: np.ndarray = scores.detach().numpy().reshape(-1, 1)
im1 = axes[1].imshow(sim_display, cmap="RdBu_r", aspect="auto", vmin=sim_display.min(), vmax=sim_display.max())
axes[1].set_xticks([])
axes[1].set_yticks(range(N_SLOTS))
axes[1].set_title("Raw similarity scores\nq @ K.T / sqrt(D_K)")
plt.colorbar(im1, ax=axes[1])

# retrieved value vector (right): should show the pulse pattern from slot 3, slightly diluted by soft weights
# RdBu: Red = +1 (first half), Blue = -1 (second half) — the stored pulse pattern
im2 = axes[2].imshow(read_result.detach().numpy().reshape(1, -1), cmap="RdBu_r", aspect="auto", vmin=-1, vmax=1)
axes[2].set_yticks([])
axes[2].set_xlabel("Value dimension")
axes[2].set_title(f"Retrieved vector r  (D_V={D_V})\nweighted sum of V rows")
plt.colorbar(im2, ax=axes[2])
# The retrieved result will show the same pulse but slightly washed out 
# (because the softmax weights aren't perfectly one-hot — the remaining ~30% 
# of attention weight goes to noise slots).
# The soft blend slightly dilutes the retrieved content, and this is the reason 
# a trained reading head learns to make its attention weights sharp.

plt.suptitle("Content-based read:  r = softmax(q @ K.T / √D_K) @ V", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

top_slot: int = int(weights.argmax().item())
print(f"Strongest weight on slot {top_slot}  (planted at slot {slot_to_write}): {weights[top_slot]:.3f}")
print(f"Read result shape: {read_result.shape}")
print(f"The retrieved vector shows the pulse pattern from slot {slot_to_write}, diluted slightly by soft weights on other slots.")

<details>
<summary>Details: <strong>Left - Middle - Right</strong></summary>

- Left: the highest bar (#3) indicates maximum attention weight
- Middle: the red row (#3) indicates maximum similarity
- Right: The same pulse (reddish at #0-7), but washed out because of softmax attention. A training is needed for making attention sharp.
</details>

<br>

<details>
<summary>Details: <strong>Modern Hopfield networks — attention is energy minimization</strong></summary>

In 2020, Ramsauer et al. showed that the scaled dot-product attention update rule is mathematically equivalent to one step of energy minimization in a *modern Hopfield network*. The stored patterns are the rows of the key matrix K, and the retrieval cue is the query q. The softmax ensures the system converges toward the stored key most similar to the query, rather than getting stuck in spurious attractors.

The key insight is that attention was already doing associative memory retrieval — the transformer community just did not name it that way. When you run a softmax over dot products between a query and a set of keys, you are performing one iteration of a content-addressable memory lookup. This reframing is not merely academic: it explains *why* attention generalizes so well, and it directly motivates the NTM architecture we build in notebook5c — which simply adds explicit write operations on top of the same K/V structure.

</details>

# Section 3 — Writing to Memory - Content-Based Addressing

Reading is a soft lookup. Writing is the inverse: given a new key and value, update the memory matrices.

The challenge:

- writing must be **differentiable** (to be trainable), 
- and **soft** — not "overwrite slot 3," but "distribute this update across all slots, weighted by how much each slot should absorb it."

To be differentiable The write operation has two stages:

1. **Erase**: reduce existing content in addressed slots — `V_new[i] = V[i] * (1 - w[i] * e)` where `e` is a learned erase vector
2. **Add**: inject new content — `V_new[i] = V_new[i] + w[i] * a` where `a` is a learned add vector

The write weights `w` are the same attention weights as the read — computed from a write query against the key matrix. This means: **the model writes where it reads** (or explicitly learns to write elsewhere).

For the key matrix: `K_new[i] = K[i] + w[i] * k_write` — add the new key to addressed slots, proportional to write weight.

In [ ]:
# Writing to memory: erase-then-add on the value matrix, additive on the key matrix

torch.manual_seed(7)

# Memory dimensions
N_SLOTS: int = 8
D_K: int = 8
D_V: int = 16

# start with empty memory
mem_K: torch.Tensor = torch.zeros(N_SLOTS, D_K)
mem_V: torch.Tensor = torch.zeros(N_SLOTS, D_V)

# --- write operation components (all produced by controller) ---
write_query: torch.Tensor = F.normalize(torch.randn(D_K), dim=0)   # what slot to address
k_write: torch.Tensor = F.normalize(torch.randn(D_K), dim=0)       # new key to store
v_add: torch.Tensor = torch.randn(D_V)                             # new value content to add
v_erase: torch.Tensor = torch.sigmoid(torch.randn(D_V))            # erase vector ∈ (0,1)

# compute write weights from write_query vs current key matrix
# (all zeros initially → uniform weights — write spreads over all slots)
write_scores: torch.Tensor = write_query @ mem_K.T / (D_K ** 0.5)  # (N_SLOTS,)
write_weights: torch.Tensor = F.softmax(write_scores, dim=-1)      # (N_SLOTS,)

# --- erase stage: reduce existing value content ---
# w[i] * erase says "how much to remove from slot i along each value dimension"
erase_matrix: torch.Tensor = torch.outer(write_weights, v_erase)   # (N_SLOTS, D_V)
mem_V_after_erase: torch.Tensor = mem_V * (1.0 - erase_matrix)

# --- add stage: inject new value content ---
add_matrix: torch.Tensor = torch.outer(write_weights, v_add)       # (N_SLOTS, D_V)
mem_V_written: torch.Tensor = mem_V_after_erase + add_matrix

# --- update key matrix: additive, same weights ---
key_add_matrix: torch.Tensor = torch.outer(write_weights, k_write) # (N_SLOTS, D_K)
mem_K_written: torch.Tensor = mem_K + key_add_matrix

# --- visualize before / after ---
fig, axes = plt.subplots(2, 3, figsize=(16, 6))

for row, (K_mat, V_mat, label) in enumerate([
    (mem_K, mem_V, "Before write"),
    (mem_K_written, mem_V_written, "After write"),
]):
    im0 = axes[row, 0].imshow(K_mat.detach().numpy(), cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
    axes[row, 0].set_title(f"{label} — Key matrix K")
    axes[row, 0].set_xlabel("Key dim")
    axes[row, 0].set_ylabel("Slot")
    plt.colorbar(im0, ax=axes[row, 0])

    im1 = axes[row, 1].imshow(V_mat.detach().numpy(), cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
    axes[row, 1].set_title(f"{label} — Value matrix V")
    axes[row, 1].set_xlabel("Value dim")
    axes[row, 1].set_ylabel("Slot")
    plt.colorbar(im1, ax=axes[row, 1])

    axes[row, 2].bar(range(N_SLOTS), write_weights.detach().numpy(), color="steelblue")
    axes[row, 2].set_xlabel("Slot")
    axes[row, 2].set_ylabel("Write weight")
    axes[row, 2].set_title("Write weights w\n(uniform: key matrix was all-zero)")
    axes[row, 2].set_ylim(0, 0.3)

plt.suptitle(
    "Write operation: K += w ⊗ k_write,   V = V * (1 − w ⊗ e) + w ⊗ a",
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()

print("Write weights (uniform because K was all-zero before first write):")
print(write_weights.detach().numpy().round(3))
print()
print("After write, the change in V is proportional to each slot's write weight.")
print("Slot with highest weight received the most of the new content.")

<details>
<summary>Details: <strong>Why softmax and not argmax — the differentiability argument</strong></summary>

The tempting implementation is: find the slot with the highest cosine similarity and write only to that one. This is argmax — and it is not differentiable. The gradient of argmax with respect to its input is zero almost everywhere and undefined at the maximum. No gradient means the controller cannot learn *which* slot to use for *what* purpose.

Softmax is the differentiable relaxation. It produces a probability distribution that is *almost* a one-hot vector when one similarity is much larger than the others, but the gradient flows smoothly back to every similarity score. During training, the model learns to produce write queries that are very similar to the keys of the target slot — making the softmax increasingly sharp and the addressing increasingly precise.

This is the core trick that makes the entire system trainable end-to-end: replace every discrete selection with a soft, weighted combination. Read is a weighted sum. Write is a weighted update. Erase is a weighted mask. Every operation is a smooth function of the weights, and the weights are a smooth function of the query. One unbroken gradient path from output to input.

</details>

<br>

<details>
<summary>Details: <strong>Is the Memory Itself Differentiable</strong></summary>

**Option 1: Memory as external state (current implementation)**

mem_K and mem_V are mutable tensors that get updated in-place at each step. They are not registered parameters. Backprop flows through the operations (the write formula, the read formula) to train whatever network produced the write query, k_write, v_add, and v_erase — but not into the memory matrices themselves.

This is the NTM/DNC design. The memory is a scratchpad that changes during a forward pass. Gradients train the controller to use it well, but the memory itself has no persistent learned weights — it starts empty at inference time and gets written by the controller.

**Option 2: Memory as a learned parameter (nn.Parameter)**

```
mem_K = nn.Parameter(torch.zeros(N_SLOTS, D_K))
mem_V = nn.Parameter(torch.zeros(N_SLOTS, D_V))
```

Now the memory matrices are trained like any weight matrix — gradient descent updates them across the dataset. The memory becomes a fixed store baked into the model weights, like a lookup table. Reading from it is differentiable and trains both the memory content and the controller simultaneously.

This is closer to how **key-value memory networks** (Weston et al., 2015) work. The downside: the memory can't change at inference time — it's frozen like any other weight.

***Option 3: Memory as a differentiable buffer within a single sequence (TBPTT)***

Keep memory as external state (Option 1), but when unrolling the controller over a sequence of T steps, don't detach the memory between steps. The computation graph spans all T write→read cycles, and gradients flow back through time — through each write operation, back to the controller outputs that produced them.

This is how NTM is actually trained. It is expensive (the graph grows with T) and requires truncated backprop through time (TBPTT) to stay tractable.

**Which matters for notebook5b?**

The current cells demonstrate the mechanics correctly for Option 1. For the lecture, the key point to make is:

>> The memory matrices are not parameters — they're state. What gets trained is the controller that decides what to write and where. The memory is a consequence of those decisions, not a model weight.

This distinction is what makes NTM architecturally interesting and also what makes it hard to train — the controller must learn a useful write policy entirely through the indirect signal of whether subsequent reads produce useful values.

</details>

<br>

<details>
<summary>Details: <strong>What parameters are differentiable in this code?</strong></summary>

The cells demonstrate **the mechanics of read/write**, but they're missing the part that makes it a neural network: a controller that produces the write/read vectors from learned linear projections.

In a real NTM, the differentiable parameters live in the controller — a small network (typically linear or LSTM) that takes the current input and produces all the addressing vectors:

```
input x  →  [Linear layers]  →  write_query, k_write, v_add, v_erase, read_query
                ↑
         these weights are the learnable parameters
```

The read and write operations themselves contain no parameters — they're fixed formulas (softmax, outer, @). The parameters are entirely in whatever produced their inputs.

So the full differentiable graph is:

```
loss  
→  retrieved value r  
→  weights w  
→  scores (q @ K.T)  
→  read_query  
→  controller weights  ✓
→  mem_K  ✗  (externl state, not a parameter)
```
</details>

# Section 4 — Location-Based Addressing

Content-based addressing asks: "which slot is *most similar* to my query?"  
But sometimes you want to ask: "give me the slot *after the one I just used*."

This is **location-based addressing** — navigating memory by position rather than by content similarity. It is essential for any task that has sequential structure: copying, sorting, iterating over a list.

The mechanism for location-based addressing works by **convolving** the current attention weights with a learned shift kernel. The shift kernel says: "stay in place," "move one step right," or "move one step left." The convolution redistributes the weight distribution accordingly.

After shifting, the weights are sharpened with a scalar `γ ≥ 1` (a learned parameter):

```
w_shifted[i] = Σ_j  w_content[j] * shift_kernel[i - j mod N]
w_final = softmax(w_shifted ^ γ)        (element-wise power, then re-normalize)
```

Higher `γ` → more peaked distribution → more precise location addressing.

In [ ]:
# Location-based addressing: shift convolution + sharpening

torch.manual_seed(3)

N_SLOTS: int = 12
D_K: int = 8

# start from a peaked content-based weight (focused on slot 4)
w_content: torch.Tensor = torch.zeros(N_SLOTS)
w_content[4] = 3.0
w_content[3] = 0.5
w_content[5] = 0.5
w_content = F.softmax(w_content, dim=-1)

# three shift kernels: stay, shift +1, shift -1
# kernel[s] = weight on "shift by s positions" (circular)
shift_kernels: dict[str, torch.Tensor] = {
    "stay (no shift)":   torch.tensor([1.0, 0.0, 0.0]),  # all weight on offset 0
    "shift right (+1)":  torch.tensor([0.0, 1.0, 0.0]),  # all weight on offset +1
    "shift left  (−1)":  torch.tensor([0.0, 0.0, 1.0]),  # all weight on offset -1 (= +N-1)
    "mixed (0.1/0.8/0.1)": torch.tensor([0.1, 0.8, 0.1]), # mostly +1 with blur
}

def circular_shift(w: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
    """Apply NTM-style circular shift convolution."""
    N: int = w.shape[0]
    result: torch.Tensor = torch.zeros(N)
    for s, ks in enumerate(kernel):
        # offset 0 → stay, offset 1 → +1, offset 2 → -1 (= N-1)
        shift: int = s if s <= len(kernel) // 2 else s - len(kernel)
        result += ks * torch.roll(w, shifts=shift, dims=0)
    return result

gamma_values: list[float] = [1.0, 3.0, 10.0]

fig, axes = plt.subplots(len(shift_kernels), len(gamma_values) + 1,
                         figsize=(16, 10), sharey=True)

for row, (kernel_name, kernel) in enumerate(shift_kernels.items()):
    # original content weights
    axes[row, 0].bar(range(N_SLOTS), w_content.numpy(), color="steelblue")
    axes[row, 0].set_title(f"Content weights\n(input)" if row == 0 else "")
    axes[row, 0].set_ylabel(kernel_name, fontsize=8)
    axes[row, 0].set_ylim(0, 0.6)

    w_shifted: torch.Tensor = circular_shift(w_content, kernel)

    for col, gamma in enumerate(gamma_values, start=1):
        # sharpen: raise to power gamma, re-normalize
        w_sharp: torch.Tensor = w_shifted ** gamma
        w_final: torch.Tensor = w_sharp / w_sharp.sum()

        axes[row, col].bar(range(N_SLOTS), w_final.numpy(), color="darkorange")
        axes[row, col].set_title(f"γ={gamma}" if row == 0 else "")
        axes[row, col].set_ylim(0, 1.0)

# column labels
for col, label in enumerate(["Content weights →"] + [f"After shift + γ={g}" for g in gamma_values]):
    axes[-1, col].set_xlabel("Memory slot")

plt.suptitle(
    "Location-based addressing: shift convolution redistributes attention; γ sharpens it",
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()

<details>
<summary>Content vs. location — two regimes of the same weight vector</summary>

The NTM addressing pipeline blends both modes in sequence:

1. **Content-based weights** `w_c`: computed from cosine similarity between write/read query and key matrix
2. **Interpolation**: `w_g = β * w_c + (1 − β) * w_prev` — blend with previous step's weights (gate `β ∈ [0,1]`)
3. **Shift convolution**: `w_s = circular_conv(w_g, shift_kernel)`
4. **Sharpening**: `w_final[i] = w_s[i]^γ / Σ w_s[j]^γ`

Setting `β=1` and `shift_kernel=[1,0,0]` recovers pure content addressing.  
Setting `β=0` and `shift_kernel=[0,1,0]` gives pure sequential iteration.  
The model learns `β`, the shift distribution, and `γ` at each step — choosing dynamically between the two modes based on the task.

This is why NTM can learn to copy a sequence: it content-addresses the start delimiter, then iterates location-by-location through the rest.

</details>


# Section 5 — Putting It Together: One Memory Step

We now have all the primitives. A single memory step consists of:

1. **Write**: controller emits `(write_query, k_write, v_add, v_erase)` → update K and V
2. **Read**: controller emits `(read_query)` → retrieve value vector `r`
3. **Use**: controller receives `r` concatenated with its hidden state → processes next input

The memory persists across steps. Across a sequence of T inputs, the memory accumulates a history of what the controller chose to store — and the controller learns, from training signal, what is worth storing and how to address it later.

In notebook5c we build the full NTM controller loop that executes this sequence.

In [ ]:
# A complete write → read cycle on the split K/V memory

torch.manual_seed(99)

N_SLOTS: int = 8
D_K: int = 8
D_V: int = 16

mem_K: torch.Tensor = torch.zeros(N_SLOTS, D_K)
mem_V: torch.Tensor = torch.zeros(N_SLOTS, D_V)

# --- STEP 1: write a "fact" into memory ---
# controller produces: a write query, a key to store, a value to store, an erase vector
fact_key: torch.Tensor   = F.normalize(torch.tensor([1., 0., 0., 0., 0., 0., 0., 0.]), dim=0)
fact_value: torch.Tensor = torch.tensor([float(i) * 0.1 for i in range(D_V)])  # recognizable pattern
erase_vec: torch.Tensor  = torch.ones(D_V) * 0.9  # strong erase (near full overwrite)

# write weights: use fact_key as write query — all zeros in K → uniform initially
w_write: torch.Tensor = F.softmax(fact_key @ mem_K.T / (D_K ** 0.5), dim=-1)

# erase then add on V; additive on K
mem_V = mem_V * (1.0 - torch.outer(w_write, erase_vec)) + torch.outer(w_write, fact_value)
mem_K = mem_K + torch.outer(w_write, fact_key)

# --- STEP 2: read back with a similar (but not identical) query ---
read_query: torch.Tensor = F.normalize(
    torch.tensor([0.95, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]), dim=0
)
w_read: torch.Tensor    = F.softmax(read_query @ mem_K.T / (D_K ** 0.5), dim=-1)
retrieved: torch.Tensor = w_read @ mem_V  # (D_V,)

# --- visualize the full cycle ---
fig, axes = plt.subplots(1, 4, figsize=(18, 3))

im0 = axes[0].imshow(mem_K.detach().numpy(), cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
axes[0].set_title("Key matrix K\n(after write)")
axes[0].set_xlabel("Key dim"); axes[0].set_ylabel("Slot")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(mem_V.detach().numpy(), cmap="RdBu", vmin=-0.5, vmax=1.5, aspect="auto")
axes[1].set_title("Value matrix V\n(after write)")
axes[1].set_xlabel("Value dim"); axes[1].set_ylabel("Slot")
plt.colorbar(im1, ax=axes[1])

axes[2].bar(range(N_SLOTS), w_read.detach().numpy(), color="darkorange")
axes[2].set_title("Read weights w\n(query ≈ written key)")
axes[2].set_xlabel("Slot"); axes[2].set_ylabel("Weight")

axes[3].plot(retrieved.detach().numpy(), "o-", color="green", label="retrieved")
axes[3].plot(fact_value.numpy(), "s--", color="steelblue", alpha=0.5, label="original")
axes[3].set_title("Retrieved vs. original value")
axes[3].set_xlabel("Value dimension")
axes[3].legend(fontsize=8)

plt.suptitle("Write → Read cycle: store a fact, retrieve it with a similar query", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

cosine_sim: float = float(F.cosine_similarity(retrieved.unsqueeze(0), fact_value.unsqueeze(0)).item())
print(f"Cosine similarity between retrieved vector and original: {cosine_sim:.4f}")
print("(Not 1.0 — write weights were soft/uniform; NTM learns to make them sharp during training.)")